<h1>Gold Delays by District<h1>

<h2>Imports<h2>

<h3>Spark and Initialization<h3>

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

<h3>Imports<h3>

In [2]:
from datetime import datetime , timedelta
import sys
import gc
sys.path.append('../work')
from config import db_properties,DB_CONFIG,jdbc_url
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql.types import FloatType, IntegerType, BooleanType
from pyspark.sql import Row
api_key = os.getenv("WARSAW_API_KEY")
import psycopg2
from psycopg2 import sql
import pyspark.sql.functions as sf
import zipfile
import urllib.request
from pyspark.testing import assertDataFrameEqual
from pyspark.sql import functions as F
import geopandas as gpd
from shapely.geometry import Point

<h2>Delays by Stop Table<h2>

<h3>Extracting data<h3>

In [ ]:
df_bus_delays_dist = spark.read.jdbc(url=jdbc_url, table="gold.bus_delays", properties=db_properties)

<h3>Grouping and Aggregation<h3>

In [ ]:
df_bus_delays_dist = df_bus_delays_dist.withColumn("weighted_delay_minutes",F.when(F.col("stop_sequence")<=3 
                                                    ,F.col("delay_minutes")*0.5)
                                                    .otherwise(F.col("delay_minutes"))) \
                            .withColumn("weighted_delay_seconds",F.when(F.col("stop_sequence")<=3
                                                    ,F.col("delay_seconds")*0.5)
                                                    .otherwise(F.col("delay_seconds")))  

In [ ]:
df_bus_delays_dist = df_bus_delays_dist.groupBy(F.window(F.col('time_gps'), "1 minute"),
                                      "district")\
                                    .agg(
                                    F.avg("delay_minutes").alias("average_delay_minutes"),
                                    F.avg("delay_seconds").alias("average_delay_seconds"),
                                    F.max("delay_minutes").alias("max_delay_minutes"),
                                    F.min("delay_minutes").alias("min_delay_minutes"),
                                    F.count("stop_id").alias("Bus_on_stop_count"),
                                    F.avg("weighted_delay_minutes").alias("average_weighted_delay_minutes"),
                                    F.avg("weighted_delay_seconds").alias("average_weighted_delay_seconds")
                                    )\
                                    .orderBy("window", "district")

<h3>Writing to Database<h3>

In [ ]:
df_bus_delays_dist.write.jdbc(url=jdbc_url, table="delays_by_district", mode="append", properties=db_properties)

In [ ]:
spark.catalog.clearCache()
gc.collect()
spark.stop()